# Proxy Validation

**Read the request documentation to tune your model for your application**

**Request Documentation**: https://platform.openai.com/docs/api-reference/completions/create

**OpenAI Documentation**: https://platform.openai.com/docs/quickstart?context=python

**Find your keys here**: https://platform.openai.com/api-keys

**Keep an eye on your credit usage**: https://platform.openai.com/usage

## Set your GPT Key to your ENV

Uncomment this line to check your environment variables. That should be set up in .zshrc or .bashrc

In [1]:
# %env

In [2]:
%pwd

'/Users/liamroy/Documents/Studies/Monash_31194990/PHD/Studies/Study_04/LLM_vocab_optimization/scripts'

## Imports

In [3]:
import os
from openai import OpenAI
client = OpenAI()

import re

import random
import time
random.seed(time.time())

from robots_and_modules.helper_functions import llm_prompt_reply

from openpyxl import load_workbook

import uuid
uid = str(uuid.uuid4())
print(uid)

825398bf-be0b-468f-b03e-6c75a234d2fc


## Setup Cell

In [4]:
iteration_quantity = 20
gpt_model = "gpt-5.1"                    
                                        # gpt-3.5-turbo     | Use this for dev/testing
                                        # gpt-4o            | Use this when deployed, more expensive
                                        # gpt-4.1-mini      | Cheap, does the code work?
                                        # gpt-5-nano        | Use this for dev/testing
                                        # gpt-4o            | Previously used model  
                                        # gpt-5-mini        | Newest model, smaller, less expensive   ('minimal, 'low', 'medium', 'high')
                                        # gpt-5.1           | Newest model, more expensive  
reasoning_effort_level = 'medium'      # 'none'||'minimal'  ~   'low'  ~  'medium' ~ 'high' - affects prompt detail level <-- !!! depends on model
max_tokens = 1500

attempt_ID = '19'

omission_probability = 0.5

summarizer_bin = "OFF"     # "OFF" or "ON"

## Building The Prompt

In [5]:
# gpt_assistant_prompt = "You are simulating human feedback for training a robot expression model. Your responses should have variability to simulate human interpretation. Different humans often interpret a robot's pose differently in similar scenarios, so stochasticity is encouraged." 
gpt_assistant_prompt = "You are an expert robotic expression design assistant that responds in the exact format [State_Number, State_Name]."

robot_morphology = "dog-shaped quadruped"

deployment_context = f"Consider a scenario where you are collaborating with a {robot_morphology} robot to locate and pick strawberries in a strawberry patch."

state_01 = "S01: [Waiting for Input, The robot is in standby mode waiting for a command from the user]"
state_02 = "S02: [Analyzing Object, The robot is analyzing a target object in front of it on the ground]"
state_03 = "S03: [Found Object, The robot has found a target object in front of it on the ground]"
state_04 = "S04: [Needs Help, The robot is experiencing an error and needs help from the user]"
state_05 = "S05: [Confused, The robot is confused and unsure what to do]"
state_06 = "S06: [Unsure, It is unclear as the robot does not appear to be in any of the described states.]"

parameter_01 = "P01: [Body Tilt]"
parameter_01_value_strings = ["tilts its head to the left",
                              "", # Neutral, so no sentence.
                              "tilts its head to the right"]

parameter_02 = "P02: [Body Lean]"
parameter_02_value_strings = ["leans its torso backward to face the sky",
                              "", # Neutral, so no sentence.
                              "leans its torso forward to face the ground"]

parameter_03 = "P03: [Body Turn]"
parameter_03_value_strings = ["turns its entire torso to the left",
                              "", # Neutral, so no sentence.
                              "turns its entire torso to the right"]

parameter_04 = "P04: [Body Height]"
parameter_04_value_strings = ["lowers its body to the ground",
                              "", # Neutral, so no sentence.
                              "firmly raises its torso as high as it can."]

parameter_05 = "P05: [Body Direction]"
parameter_05_value_strings = ["",  # NAN or neutral, so no sentence.
                              "The robot turns its whole body to directly face the user in the scene.",
                              "The robot turns its whole body to directly face a nearby strawberry in the scene."]

parameter_06 = "P06: [Pose Duration]"
parameter_06_value_strings = ["The robot holds this pose for a short duration of 1 second.",
                              "", # Neutral, so no sentence.
                              "The robot holds this pose for a long duration of 8 seconds."]

parameter_07 = "P07: [Motion Velocity]"
parameter_07_value_strings = ["The robot moves very slowly to achieve this pose.",
                              "",  # Neutral speed, so no sentence.
                              "The robot moves quickly to achieve this pose."]

parameter_08 = "P08: [Motion Smoothness]"
parameter_08_value_strings = ["",  # NAN, so no sentence.
                              "The robot's motion is smooth without any disturbances.",
                              "The robot's motion is rough, jittery and shaky."]

In [6]:
# Helper Functions

# Function to randomly omit a parameter with a XX% chance
def maybe_include(omission_prob, param_string):
        return param_string if random.random() > omission_prob else ""


In [7]:
test_expression_string = "The robot faces a nearby strawberry in the scene. The robot tilts its torso to the left, and leans its torso forward. The robot's motion is smooth without any disturbances. The robot holds this pose for a long duration of 8 seconds."
summary_string = f"{deployment_context} Summarize the explanation below, focusing on describing the robot's actions in this scenario:\n"


summary_assistant_prompt = "You are an helpful robotic expression design assistant."
summarized_expression_string = llm_prompt_reply(
                    prompt=summary_string + test_expression_string,
                    client=client,
                    llm_model=gpt_model,
                    llm_assistant_prompt=summary_assistant_prompt,
                    max_output_tokens=max_tokens,
                    reasoning_effort=reasoning_effort_level,
                )

print(f"OG Expression String:\n{test_expression_string}\n")
print(f"Summarized Expression String:\n{summarized_expression_string}\n")


OG Expression String:
The robot faces a nearby strawberry in the scene. The robot tilts its torso to the left, and leans its torso forward. The robot's motion is smooth without any disturbances. The robot holds this pose for a long duration of 8 seconds.

Summarized Expression String:
The dog-shaped quadruped robot smoothly tilts its torso to the left and leans forward toward a nearby strawberry, then steadily maintains this pose without disturbance for about 8 seconds.



### Prompt Generating Function

In [8]:
def generate_gpt_user_prompt(parameter_values, omission_probability, summarizer_bin_toggle):

        # This is done to randomize the order in which the states are presented to GPT, to eliminate any bias for states presented first
        state_list = [state_01, state_02, state_03, state_04, state_05]
        random.shuffle(state_list)

        # Generate strings for each parameter and randomly omit with 20% probability
        P01_string = maybe_include(omission_probability, parameter_01_value_strings[0] if parameter_values[0] == "left" else parameter_01_value_strings[2] if parameter_values[0] == "right" else "")
        P02_string = maybe_include(omission_probability, parameter_02_value_strings[0] if parameter_values[1] == "backward" else parameter_02_value_strings[2] if parameter_values[1] == "forward" else "")
        P03_string = maybe_include(omission_probability, parameter_03_value_strings[0] if parameter_values[2] == "left" else parameter_03_value_strings[2] if parameter_values[2] == "right" else "")
        P04_string = maybe_include(omission_probability, parameter_04_value_strings[0] if parameter_values[3] == "low" else parameter_04_value_strings[2] if parameter_values[3] == "high" else "")
        P05_string = maybe_include(omission_probability, parameter_05_value_strings[1] if parameter_values[4] == "user" else parameter_05_value_strings[2] if parameter_values[4] == "object" else "")
        P06_string = maybe_include(omission_probability, parameter_06_value_strings[0] if parameter_values[5] == "short" else parameter_06_value_strings[2] if parameter_values[5] == "long" else "")
        P07_string = maybe_include(omission_probability, parameter_07_value_strings[0] if parameter_values[6] == "slow" else parameter_07_value_strings[2] if parameter_values[6] == "fast" else "")
        P08_string = maybe_include(omission_probability, parameter_08_value_strings[1] if parameter_values[7] == "smooth" else parameter_08_value_strings[2] if parameter_values[7] == "shaky" else "")

        # Create a list of pose-related strings (P01 to P04)
        pose_strings = [P01_string, P02_string, P03_string, P04_string]

        # Filter out empty strings and format the sentence with commas and 'and' before the last item
        pose_strings = [pose for pose in pose_strings if pose]  # Remove empty strings
        if len(pose_strings) > 1:
                body_movement_sentence = ", ".join(pose_strings[:-1]) + ", and " + pose_strings[-1] + "."
        elif pose_strings:
                body_movement_sentence = pose_strings[0] + "."
        else:
                body_movement_sentence = ""

        # Add "The robot" at the beginning only if there's any body movement
        if body_movement_sentence:
                body_movement_sentence = "The robot " + body_movement_sentence

        # Create a coherent paragraph by joining all parts
        sentence_parts = [
                P05_string,  # Direction is always first
                body_movement_sentence,  # Fluid body movement sentence
                P08_string,  # Motion smoothness
                P07_string,  # Motion speed
                P06_string,  # Pose duration
        ]

        # Filter out empty strings and join them into a readable paragraph
        expression_string = " ".join([part for part in sentence_parts if part])


        if summarizer_bin_toggle == "ON":
                # Creatively summarize the expression string 
                print("Summarization is ON, summarizing expression string.")
                summary_string = f"{deployment_context} Summarize the explanation below, focusing on describing the robot's actions in this scenario:\n"

                summarized_expression_string = llm_prompt_reply(
                        prompt=summary_string + expression_string,
                        client=client,
                        llm_model=gpt_model,
                        llm_assistant_prompt=summary_assistant_prompt,
                        max_output_tokens=max_tokens,
                        reasoning_effort=reasoning_effort_level,
                        )

        elif summarizer_bin_toggle == "OFF":
                # No summarization, just use the original expression string
                print("Summarization is OFF, using original expression string.")
                summarized_expression_string = expression_string


        task_id = str(uuid.uuid4())

        return f'''

Task ID: {task_id}

{gpt_assistant_prompt}

{deployment_context}

In this scenario, the robot can be in one of 5 possible states. The data in this list is in the format: "State Number: [State Name, State Description]"

{state_list[0]}

{state_list[1]}

{state_list[2]}

{state_list[3]}

{state_list[4]}

{state_06}

—————————

Below is a description of the robot using its body pose to express its internal state:

{summarized_expression_string}

—————————

Your task:

Please estimate what state you think the robot is in based on this description. If none of the states seem to match the description, select 'Unsure'.

Your response MUST be a single line in the exact format shown below (see example and reference):

[State_Number, State_Name]

Reference: 
State_Number = number of the selected robot state (e.g. S04)
State_Name = name of the selected robot state (e.g. Progressing)
'''

### Test Prompt

In [9]:
test_parameter_values = ["left", 
                         "forward",	
                         "neutral", 
                         "neutral", 
                         "object", 
                         "long", 
                         "medium", 
                         "smooth"]

test_omission_probability = 0.5

test_prompt = generate_gpt_user_prompt(test_parameter_values, test_omission_probability, summarizer_bin)

print(test_prompt)

Summarization is OFF, using original expression string.


Task ID: 7dc9c82a-5493-4455-8030-e95ad80bd30e

You are an expert robotic expression design assistant that responds in the exact format [State_Number, State_Name].

Consider a scenario where you are collaborating with a dog-shaped quadruped robot to locate and pick strawberries in a strawberry patch.

In this scenario, the robot can be in one of 5 possible states. The data in this list is in the format: "State Number: [State Name, State Description]"

S02: [Analyzing Object, The robot is analyzing a target object in front of it on the ground]

S03: [Found Object, The robot has found a target object in front of it on the ground]

S05: [Confused, The robot is confused and unsure what to do]

S04: [Needs Help, The robot is experiencing an error and needs help from the user]

S01: [Waiting for Input, The robot is in standby mode waiting for a command from the user]

S06: [Unsure, It is unclear as the robot does not appear to be in an

## Generate Your Prompt

### Multi-Iteration

This cell is running the following...

* Iterates through all conditions/states
* Loads that condition/state parameter combination from workbook
* Tests the given combination against the LLM framework
* Logs the results in a new sheet in workbook

In [10]:
row_counter = 1

study_condition = ['LLM', 'HUM', 'RAN']
real_state = ['WFI', 'AO', 'FO', 'NH', 'C']

for condition in study_condition:
    for state in real_state:
        
        row_counter += 1 
        expression_ID = condition + "_" + state

        ## LOAD THE DATA FOR SPECIFIC CONDITION

        workbook_path = "./../data/proxy_validation/proxy_validation.xlsx"
        response_book = load_workbook(workbook_path, data_only=True)
        load_sheet = response_book['HUMAN_ACCURACY']

        # Initialize an empty list for concatenated values
        parameter_list = []     

        # Iterate through the cells Q2 to X2 (columns 17 to 24)
        for col in range(17, 25):  # openpyxl uses 1-based indexing for rows and columns
            cell_value = load_sheet.cell(row=row_counter, column=col).value
            if cell_value is not None:
                parameter_list.append(str(cell_value))  # Convert each cell value to string and append to the list

        
        ## TEST AND LOG THE DATA FOR SPECIFIC CONDITION
        sheet_name = expression_ID + '_' + attempt_ID 

        # Enter the data in spreadsheet format
        workbook_path = "./../data/proxy_validation/proxy_validation.xlsx"
        response_book = load_workbook(workbook_path)

        try: # Try to open existing sheet
            response_sheet = response_book[sheet_name]
        except KeyError:  # If ot doesn't exist. create it
            response_sheet = response_book.create_sheet(title=sheet_name)
        response_sheet["A1"] = "model"
        response_sheet["B1"] = "study cond"
        response_sheet["C1"] = "real state"
        response_sheet["D1"] = "iteration"
        response_sheet["E1"] = "state number"
        response_sheet["F1"] = "state name"
        response_sheet["G1"] = "justification"
        response_sheet["H1"] = "P1 Tilt"
        response_sheet["I1"] = "P2 Lean"
        response_sheet["J1"] = "P3 Turn"
        response_sheet["K1"] = "P4 Body Height"
        response_sheet["L1"] = "P5 Direction"
        response_sheet["M1"] = "P6 Duration"
        response_sheet["N1"] = "P7 Velocity"
        response_sheet["O1"] = "P8 Smoothness"


        response_sheet["U1"] = "llm_model"
        response_sheet["U2"] = gpt_model

        response_sheet["V1"] = "reasoning effort"
        response_sheet["V2"] = reasoning_effort_level

        response_sheet["W1"] = "summarizer"
        response_sheet["W2"] = summarizer_bin

        response_sheet["X1"] = "omission_prob"
        response_sheet["X2"] = omission_probability

        response_book.save(workbook_path)

        print(f'\n************************************')
        print(f'********* expression ID: {expression_ID}')
        print(f'************************************\n')

        error_counter = 0

        for iteration in range(0, iteration_quantity):
            print(f'~~~~~~~~~~~~ Iteration {iteration:02d}')

            # adding to excel
            response_sheet["A"+str(iteration+2)] = gpt_model
            response_sheet["B"+str(iteration+2)] = condition
            response_sheet["C"+str(iteration+2)] = state
            response_sheet["D"+str(iteration+2)] = iteration

            

            completion = llm_prompt_reply(
                    prompt=generate_gpt_user_prompt(parameter_list, omission_probability, summarizer_bin),
                    client=client,
                    llm_model=gpt_model,
                    llm_assistant_prompt=gpt_assistant_prompt,
                    max_output_tokens=max_tokens,
                    reasoning_effort=reasoning_effort_level,
                )


            print(f"\ncompletion before strip: {completion}\n")

            # --- NEW: parse the single response ---
            # completion might be like "[S02, Analyzing Object]" or include extra text/newlines
            text = completion.strip()

            # Find first bracketed pair anywhere in the text
            match = re.search(r"\[\s*([A-Za-z0-9_]+)\s*,\s*(.+?)\s*\]", text)

            if match:
                state_number, state_name = match.groups()
            else:
                print(f"ERROR at iteration {iteration}: No match found in: {text}")
                error_counter += 1
                state_number, state_name = "NaN", "NaN"   # or continue / skip, your call

            print(f'Appending: {state_number}, {state_name}')

            row = str(iteration + 2)
            response_sheet["E" + row] = state_number
            response_sheet["F" + row] = state_name
            response_sheet["G" + row] = 'NaN'

            response_sheet["H" + row] = parameter_list[0]
            response_sheet["I" + row] = parameter_list[1]
            response_sheet["J" + row] = parameter_list[2]
            response_sheet["K" + row] = parameter_list[3]
            response_sheet["L" + row] = parameter_list[4]
            response_sheet["M" + row] = parameter_list[5]
            response_sheet["N" + row] = parameter_list[6]
            response_sheet["O" + row] = parameter_list[7]

            print('\n')

        print(f'completed {iteration+1} iterations with {error_counter} match errors')
            
        response_book.save(workbook_path)

print("\n*********")
print("ALL DONE!")
print("*********")




************************************
********* expression ID: LLM_WFI
************************************

~~~~~~~~~~~~ Iteration 00
Summarization is OFF, using original expression string.

completion before strip: [S01, Waiting for Input]

Appending: S01, Waiting for Input


~~~~~~~~~~~~ Iteration 01
Summarization is OFF, using original expression string.

completion before strip: [S01, Waiting for Input]

Appending: S01, Waiting for Input


~~~~~~~~~~~~ Iteration 02
Summarization is OFF, using original expression string.

completion before strip: [S01, Waiting for Input]

Appending: S01, Waiting for Input


~~~~~~~~~~~~ Iteration 03
Summarization is OFF, using original expression string.

completion before strip: [S01, Waiting for Input]

Appending: S01, Waiting for Input


~~~~~~~~~~~~ Iteration 04
Summarization is OFF, using original expression string.

completion before strip: [S06, Unsure]

Appending: S06, Unsure


~~~~~~~~~~~~ Iteration 05
Summarization is OFF, using original 

### Single Call

In [ ]:
# SETUP

parameter_list = [
        "left",
        "backward",
        "left",
        "low",
        "user",
        "short",
        "fast",
        "shaky"]

omission_probability = 0.5

In [ ]:

completion = llm_prompt_reply(
        prompt=generate_gpt_user_prompt(parameter_list, omission_probability, summarizer_bin),
        client=client,
        llm_model=gpt_model,
        llm_assistant_prompt=gpt_assistant_prompt,
        max_output_tokens=max_tokens,
        reasoning_effort=reasoning_effort_level,
    )


print('Model Output:\n')
print(completion)

In [ ]:
print(f'''
      Parameter Config:\n
P01_value = {parameter_list[0]} Body Tilt
P02_value = {parameter_list[1]} Body Lean
P03_value = {parameter_list[2]} Body Turn
P04_value = {parameter_list[3]} Body Height
P05_value = {parameter_list[4]} Body Direction
P06_value = {parameter_list[5]} Pose Duration
P07_value = {parameter_list[6]} Motion Velocity
P08_value = {parameter_list[7]} Motion Smoothness
''')


print(completion.choices[0].message.content)